# Module 1: BPE Tokenizer

Tokenization is the first and crucial step in any LLM pipeline. In this notebook, we'll explore:

## What You'll Learn

- **What is tokenization?** Converting text to numbers that models can process
- **Byte Pair Encoding (BPE)** - The algorithm behind GPT tokenizers
- **Training a tokenizer** - Using RustBPE for fast training
- **Efficient inference** - Using tiktoken for production
- **Special tokens** - Conversation structure (`<|user_start|>`, `<|assistant_end|>`, etc.)

## Why This Matters

- **Vocabulary size** affects model capacity and training efficiency
- **Tokenization quality** impacts how well the model understands language
- **Speed** matters for both training and inference

## 1.1 The Tokenization Problem

Neural networks work with numbers, not text. We need to convert:

```
"Hello, world!" → [15496, 11, 995, 0]
```

But how do we decide what becomes a token?

In [ ]:
# The naive approach: character-level tokenization
text = "Hello, world!"

# Each character becomes a token
char_tokens = list(text)
char_ids = [ord(c) for c in text]

print(f"Text: {text}")
print(f"Characters: {char_tokens}")
print(f"Character IDs: {char_ids}")
print(f"Vocabulary size: 256 (all possible bytes)")
print(f"Sequence length: {len(char_tokens)}")

### Problems with Character-Level Tokenization

1. **Long sequences**: "Hello, world!" → 13 tokens
2. **No semantic meaning**: The model has to learn that 'H', 'e', 'l', 'l', 'o' forms "Hello"
3. **Inefficient attention**: Attention is O(n²), so longer sequences are expensive

In [ ]:
# The other extreme: word-level tokenization
text = "Hello, world! This is tokenization."
word_tokens = text.split()

print(f"Text: {text}")
print(f"Word tokens: {word_tokens}")
print(f"Sequence length: {len(word_tokens)}")

### Problems with Word-Level Tokenization

1. **Huge vocabulary**: English has 170,000+ words
2. **Out-of-vocabulary (OOV)**: What about "ChatGPT" or "nanochat"?
3. **Morphology lost**: "running", "runs", "ran" are all different tokens

**Solution: Subword tokenization with BPE!**

## 1.2 Byte Pair Encoding (BPE) Algorithm

BPE finds a middle ground by:
1. Starting with bytes (256 tokens)
2. Iteratively merging the most frequent pairs
3. Building a vocabulary of common subwords

Let's implement it from scratch!

In [ ]:
from collections import Counter

def get_pair_counts(token_sequences):
    """Count all adjacent pairs in the token sequences."""
    pairs = Counter()
    for seq in token_sequences:
        for i in range(len(seq) - 1):
            pairs[(seq[i], seq[i+1])] += 1
    return pairs

def merge_pair(token_sequences, pair, new_token):
    """Replace all occurrences of pair with new_token."""
    new_sequences = []
    for seq in token_sequences:
        new_seq = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i+1]) == pair:
                new_seq.append(new_token)
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        new_sequences.append(new_seq)
    return new_sequences

def train_bpe(text, num_merges):
    """Train a simple BPE tokenizer."""
    # Start with bytes
    token_sequences = [[b for b in text.encode('utf-8')]]
    merges = {}  # (pair) -> new_token
    vocab = {i: bytes([i]) for i in range(256)}  # Initial vocabulary
    
    for i in range(num_merges):
        pairs = get_pair_counts(token_sequences)
        if not pairs:
            break
        
        # Find most frequent pair
        best_pair = max(pairs, key=pairs.get)
        new_token = 256 + i
        
        # Record the merge
        merges[best_pair] = new_token
        vocab[new_token] = vocab[best_pair[0]] + vocab[best_pair[1]]
        
        # Apply the merge
        token_sequences = merge_pair(token_sequences, best_pair, new_token)
        
        print(f"Merge {i+1}: {vocab[best_pair[0]]} + {vocab[best_pair[1]]} → {vocab[new_token]} (count: {pairs[best_pair]})")
    
    return merges, vocab, token_sequences[0]

# Train on a simple example
text = "the cat sat on the mat. the cat is fat."
merges, vocab, tokens = train_bpe(text, num_merges=10)

print(f"\nFinal tokens: {tokens}")
print(f"Decoded: {''.join(vocab[t].decode('utf-8', errors='replace') for t in tokens)}")
print(f"Compression: {len(text.encode('utf-8'))} bytes → {len(tokens)} tokens")

## 1.3 GPT-4 Style Tokenization

GPT-4's tokenizer adds an important twist: **pre-tokenization with regex**.

Before BPE, the text is split using a regex pattern that separates:
- Words with apostrophes: `'s`, `'t`, `'re`, `'ve`, `'m`, `'ll`, `'d`
- Letters vs numbers vs punctuation
- Whitespace handling

In [ ]:
import regex  # pip install regex (for Unicode support)

# This is the split pattern used by GPT-4 (and nanochat)
# Note: nanochat uses \p{N}{1,2} instead of \p{N}{1,3} to save vocabulary space
SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

text = "Hello, I'm learning about tokenization! The year is 2024."
pre_tokens = regex.findall(SPLIT_PATTERN, text)

print(f"Original: {text}")
print(f"Pre-tokens: {pre_tokens}")
print(f"\nNotice how:")
print("  - 'I'm' is split into 'I' and "'m'")
print("  - Numbers like '2024' become '20' and '24' (limited to 2 digits)")
print("  - Punctuation is separated from words")

## 1.4 Nanochat's Tokenizer Implementation

Nanochat uses two tokenizer implementations:

1. **RustBPE** - Fast training in Rust
2. **tiktoken** - Efficient inference from OpenAI

Let's explore the actual nanochat tokenizer!

In [ ]:
# First, let's look at nanochat's special tokens
SPECIAL_TOKENS = [
    "<|bos|>",           # Beginning of sequence
    "<|user_start|>",    # User message starts
    "<|user_end|>",      # User message ends
    "<|assistant_start|>",  # Assistant message starts
    "<|assistant_end|>",    # Assistant message ends
    "<|python_start|>",  # Tool: Python code starts
    "<|python_end|>",    # Tool: Python code ends
    "<|output_start|>",  # Tool output starts
    "<|output_end|>",    # Tool output ends
]

print("Nanochat Special Tokens:")
print("="*50)
for i, token in enumerate(SPECIAL_TOKENS):
    print(f"{i}: {token}")

print("\n" + "="*50)
print("These tokens structure conversations like:")
print("""
<|bos|><|user_start|>What is 2+2?<|user_end|>
<|assistant_start|>Let me calculate: <|python_start|>2+2<|python_end|>
<|output_start|>4<|output_end|>
The answer is 4.<|assistant_end|>
""")

In [ ]:
# Let's use a pre-trained tokenizer to see how it works
import tiktoken

# Load GPT-2's tokenizer as an example
enc = tiktoken.get_encoding("gpt2")

text = "Hello, world! I'm learning about tokenization."
tokens = enc.encode(text)

print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")
print(f"Vocabulary size: {enc.n_vocab}")

# Let's see what each token represents
print("\nToken breakdown:")
for token in tokens:
    decoded = enc.decode([token])
    print(f"  {token:5d} → '{decoded}'")

In [ ]:
# Compare compression across different texts
test_texts = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "def hello_world():\n    print('Hello!')",
    "π ≈ 3.14159265358979",
    "你好，世界！",  # Chinese
]

print("Tokenization efficiency:")
print("="*70)
print(f"{'Text':<40} {'Bytes':>8} {'Tokens':>8} {'Ratio':>8}")
print("-"*70)

for text in test_texts:
    num_bytes = len(text.encode('utf-8'))
    num_tokens = len(enc.encode(text))
    ratio = num_bytes / num_tokens
    display_text = text[:37] + "..." if len(text) > 40 else text
    print(f"{display_text:<40} {num_bytes:>8} {num_tokens:>8} {ratio:>7.2f}x")

print("\nHigher ratio = better compression (more bytes per token)")

## 1.5 Training a Tokenizer with RustBPE

Now let's see how nanochat trains its tokenizer. The process:

1. Download training text (FinWeb dataset)
2. Train BPE with RustBPE (fast Rust implementation)
3. Export to tiktoken format for efficient inference

In [ ]:
# Simulate tokenizer training with a small dataset
# In practice, nanochat trains on 2 billion characters!

training_text = """
The Transformer architecture has revolutionized natural language processing.
It uses self-attention mechanisms to process sequences in parallel.
Models like GPT and BERT are based on this architecture.
GPT uses a decoder-only transformer for autoregressive text generation.
BERT uses an encoder-only transformer for understanding tasks.
""" * 100  # Repeat to get more training data

print(f"Training data size: {len(training_text):,} characters")

# In the actual nanochat code, this would be:
# from nanochat.tokenizer import RustBPETokenizer
# tokenizer = RustBPETokenizer.train_from_iterator([training_text], vocab_size=65536)

print("\nNanochat tokenizer settings:")
print(f"  - Vocabulary size: 65,536 (2^16)")
print(f"  - Special tokens: 9 (BOS, user/assistant markers, tool markers)")
print(f"  - Training data: ~2 billion characters from FinWeb")
print(f"  - Training time: ~30 minutes on a single core")

## 1.6 Conversation Rendering

A key feature of nanochat's tokenizer is rendering conversations with proper structure and training masks.

In [ ]:
# Let's simulate how nanochat renders a conversation

conversation = {
    "messages": [
        {"role": "user", "content": "What is the capital of France?"},
        {"role": "assistant", "content": "The capital of France is Paris."},
        {"role": "user", "content": "What's its population?"},
        {"role": "assistant", "content": "Paris has a population of about 2.2 million."}
    ]
}

# This is what the rendered conversation looks like:
rendered = """
<|bos|><|user_start|>What is the capital of France?<|user_end|>
<|assistant_start|>The capital of France is Paris.<|assistant_end|>
<|user_start|>What's its population?<|user_end|>
<|assistant_start|>Paris has a population of about 2.2 million.<|assistant_end|>
"""

print("Rendered conversation:")
print(rendered)

# The mask indicates which tokens to train on (1 = train, 0 = don't train)
print("\nTraining mask logic:")
print("  - User messages: mask = 0 (don't predict user input)")
print("  - Assistant messages: mask = 1 (train to generate these)")
print("  - Special tokens: mask = 0 or 1 depending on context")
print("  - Tool outputs: mask = 0 (comes from external tool)")

In [ ]:
# Visualize the mask
# In the actual tokenizer.py, there's a visualize_tokenization function

def visualize_mask(tokens_text, mask):
    """Visualize which parts are trained (green) vs not trained (red)."""
    RED = '\033[91m'
    GREEN = '\033[92m'
    RESET = '\033[0m'
    
    result = []
    for token, m in zip(tokens_text, mask):
        color = GREEN if m == 1 else RED
        result.append(f"{color}{token}{RESET}")
    return '|'.join(result)

# Simplified example
tokens_text = ["<|bos|>", "<|user_start|>", "Hello", "<|user_end|>", 
               "<|assistant_start|>", "Hi", "there", "!", "<|assistant_end|>"]
mask = [0, 0, 0, 0, 0, 1, 1, 1, 1]

print("Token visualization (RED = no train, GREEN = train):")
print(visualize_mask(tokens_text, mask))

## 1.7 Why Vocabulary Size Matters

The choice of vocabulary size involves tradeoffs:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

vocab_sizes = [256, 1024, 4096, 16384, 65536, 262144]

# Rough estimates for a 1B parameter model
embedding_params = [v * 768 for v in vocab_sizes]  # Assuming 768-dim embeddings
seq_compression = [1, 2, 3, 4, 5, 6]  # Approximate bytes per token

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(len(vocab_sizes)), [p/1e6 for p in embedding_params])
ax1.set_xticks(range(len(vocab_sizes)))
ax1.set_xticklabels([f"{v//1024}K" if v >= 1024 else str(v) for v in vocab_sizes])
ax1.set_xlabel('Vocabulary Size')
ax1.set_ylabel('Embedding Parameters (Millions)')
ax1.set_title('Cost: More Parameters')

ax2.bar(range(len(vocab_sizes)), seq_compression)
ax2.set_xticks(range(len(vocab_sizes)))
ax2.set_xticklabels([f"{v//1024}K" if v >= 1024 else str(v) for v in vocab_sizes])
ax2.set_xlabel('Vocabulary Size')
ax2.set_ylabel('Bytes per Token')
ax2.set_title('Benefit: Better Compression')

plt.tight_layout()
plt.show()

print("\nNanochat uses 65,536 (2^16) tokens:")
print("  - Good compression ratio")
print("  - Reasonable embedding table size")
print("  - Power of 2 for efficient GPU memory alignment")

## 1.8 Hands-On Exercise: Analyze Tokenization

Try tokenizing different types of text and observe the patterns!

In [ ]:
# Exercise: Analyze how different text types tokenize

def analyze_tokenization(text, enc):
    tokens = enc.encode(text)
    print(f"Text: {text}")
    print(f"Tokens ({len(tokens)}): {tokens}")
    print("Breakdown:")
    for t in tokens:
        print(f"  {t:5d} → '{enc.decode([t])}'")
    print(f"Compression: {len(text.encode('utf-8'))} bytes → {len(tokens)} tokens ({len(text.encode('utf-8'))/len(tokens):.2f} bytes/token)")
    print()

enc = tiktoken.get_encoding("gpt2")

# Try these examples
examples = [
    "Hello",
    "Hello!!!",
    "   Hello   ",  # Extra whitespace
    "12345",
    "3.14159",
    "def f(x): return x*2",
]

for ex in examples:
    analyze_tokenization(ex, enc)

## Summary

In this notebook, we learned:

1. ✅ **Why tokenization matters** - Converting text to numbers efficiently
2. ✅ **BPE algorithm** - Iterative merging of frequent byte pairs
3. ✅ **GPT-4 style pre-tokenization** - Regex-based splitting before BPE
4. ✅ **Nanochat's tokenizer** - RustBPE + tiktoken + special tokens
5. ✅ **Conversation rendering** - Structuring chat data with masks
6. ✅ **Vocabulary size tradeoffs** - Compression vs parameter count

## Key Takeaways

- **65,536 tokens** is a good balance for small-to-medium LLMs
- **Pre-tokenization** with regex improves tokenization quality
- **Special tokens** structure conversations and enable tool use
- **Training masks** ensure we only train on assistant outputs

## Next Steps

Continue to **[Module 2: GPT Architecture](02_gpt_architecture.ipynb)** to learn:
- Transformer architecture fundamentals
- Rotary Position Embeddings (RoPE)
- Multi-Query Attention (MQA)
- Modern improvements: QK-Norm, ReLU², soft-capped logits

---

**Estimated time for this notebook: 30-45 minutes**